# INC-002 Walking Skeleton Review

This notebook inspects an existing INC-002 run. It does not rerun or mutate the supplied database. The idempotency section makes an isolated copy of that database, reruns the pipeline against the copy, and compares table contents and report bytes. Raw inputs are read only.

Set `INC002_RUN_DIR` to the output directory used by the CLI if it differs from the default. The default is `/tmp/inc002-review-20261012`. The `INC002_DATA_DIR` variable can override the repository `data/` directory.

## Review Sections
1. Load environment, paths, DuckDB connection, and display helpers.
2. Record run metadata and expected input sources.
3. Check raw manifest coverage, source counts, missing files, and duplicate paths.
4. Compare current raw file sizes and hashes with the manifest snapshot.
5. Check silver schemas, grains, key uniqueness, nulls, and score distributions.
6. Review quality summaries and traceable reject references.
7. Check daily gold grain, coverage, null patterns, and metric summaries.
8. Inspect gold key constraints and MERGE-related key invariants.
9. Re-run against an isolated copy and compare all persisted tables and report bytes.
10. Summarize acceptance checks with PASS/FAIL/NOT RUN evidence.

Only aggregate outputs and one explicitly selected example are displayed; reject raw-value payloads are not printed.

In [ ]:
import hashlib
import html
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
from IPython.display import HTML, Markdown, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "wearable_pipeline").is_dir()
)
DATA_DIR = Path(os.environ.get("INC002_DATA_DIR", ROOT / "data")).resolve()
RUN_DIR = Path(os.environ.get("INC002_RUN_DIR", "/tmp/inc002-review-20261012")).resolve()
DATABASE_PATH = RUN_DIR / "wearable.duckdb"
REPORT_PATH = RUN_DIR / "quality" / "report.md"

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        f"DuckDB file not found: {DATABASE_PATH}. Set INC002_RUN_DIR to the CLI output directory."
    )
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {DATA_DIR}")

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
acceptance = {}


def render_table(title, columns, rows):
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<h4>{html.escape(title)}</h4><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_query(title, sql, parameters=None, db=connection):
    cursor = db.execute(sql, parameters or [])
    render_table(title, [item[0] for item in cursor.description], cursor.fetchall())


report_text = REPORT_PATH.read_text(encoding="utf-8")
run_id = next(
    (line.split(": ", 1)[1] for line in report_text.splitlines() if line.startswith("Run id: ")),
    None,
)
print(f"Database: {DATABASE_PATH}")
print(f"Data directory: {DATA_DIR}")
print(f"Run ID: {run_id}")
print(f"DuckDB version: {duckdb.__version__}")
acceptance["CLI database and report available"] = DATABASE_PATH.is_file() and REPORT_PATH.is_file() and run_id is not None

In [ ]:
# Sections 2-3: run metadata, expected files, manifest completeness, and per-source file counts.
expected_paths = {
    path.relative_to(DATA_DIR).as_posix()
    for path in DATA_DIR.rglob("*")
    if path.is_file()
}
manifest_rows = connection.execute(
    "SELECT relative_path, source, size_bytes, sha256, record_count "
    "FROM raw_manifest ORDER BY relative_path"
).fetchall()
manifest_paths = [row[0] for row in manifest_rows]
manifest_path_set = set(manifest_paths)
missing_from_manifest = sorted(expected_paths - manifest_path_set)
unexpected_in_manifest = sorted(manifest_path_set - expected_paths)
duplicate_manifest_paths = len(manifest_paths) - len(manifest_path_set)

show_query(
    "Manifest counts by source",
    """SELECT source, COUNT(*) AS files, SUM(size_bytes) AS bytes,
              SUM(record_count) AS csv_records
       FROM raw_manifest GROUP BY source ORDER BY source""",
)
render_table(
    "Manifest coverage checks",
    ["check", "observed", "status"],
    [
        ("Filesystem files", len(expected_paths), "PASS" if len(expected_paths) == len(manifest_paths) else "FAIL"),
        ("Manifest rows", len(manifest_paths), "PASS" if not missing_from_manifest and not unexpected_in_manifest else "FAIL"),
        ("Duplicate paths", duplicate_manifest_paths, "PASS" if duplicate_manifest_paths == 0 else "FAIL"),
        ("device_metadata.csv catalogued", "health_summaries/device_metadata.csv" in manifest_path_set,
         "PASS" if "health_summaries/device_metadata.csv" in manifest_path_set else "FAIL"),
    ],
)
print("Missing manifest paths:", missing_from_manifest[:10] or "none")
print("Unexpected manifest paths:", unexpected_in_manifest[:10] or "none")
acceptance["Manifest covers current raw tree exactly"] = (
    expected_paths == manifest_path_set and duplicate_manifest_paths == 0
)
acceptance["Device metadata is catalogued"] = "health_summaries/device_metadata.csv" in manifest_path_set

In [ ]:
# Section 4: compare current raw-file bytes with the size/hash snapshot captured by the run.
manifest_by_path = {row[0]: row for row in manifest_rows}
size_mismatches = []
hash_mismatches = []
for relative_path in sorted(expected_paths):
    path = DATA_DIR / relative_path
    expected_row = manifest_by_path.get(relative_path)
    if expected_row is None:
        continue
    content = path.read_bytes()
    if len(content) != expected_row[2]:
        size_mismatches.append(relative_path)
    if hashlib.sha256(content).hexdigest() != expected_row[3]:
        hash_mismatches.append(relative_path)

render_table(
    "Raw immutability snapshot checks",
    ["check", "mismatches", "status"],
    [
        ("File sizes", len(size_mismatches), "PASS" if not size_mismatches else "FAIL"),
        ("SHA-256 hashes", len(hash_mismatches), "PASS" if not hash_mismatches else "FAIL"),
    ],
)
print("Size mismatch examples:", size_mismatches[:5] or "none")
print("Hash mismatch examples:", hash_mismatches[:5] or "none")
acceptance["Current raw files match the run manifest"] = not size_mismatches and not hash_mismatches

In [ ]:
# Section 5: inspect in-scope silver schemas, key nulls/duplicates, and normalized score ranges.
show_query(
    "Silver table schemas",
    """SELECT table_name, ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name IN ('silver_participants', 'silver_wellness_survey')
       ORDER BY table_name, ordinal_position""",
)
show_query(
    "Silver grain and key checks",
    """SELECT 'silver_participants' AS table_name, COUNT(*) AS rows,
              COUNT(*) FILTER (WHERE participant_id IS NULL) AS null_key_rows,
              (SELECT COUNT(*) FROM (
                  SELECT participant_id FROM silver_participants
                  GROUP BY participant_id HAVING COUNT(*) > 1
              )) AS duplicate_keys
       FROM silver_participants
       UNION ALL
       SELECT 'silver_wellness_survey', COUNT(*),
              COUNT(*) FILTER (WHERE participant_id IS NULL OR local_date IS NULL),
              (SELECT COUNT(*) FROM (
                  SELECT participant_id, local_date FROM silver_wellness_survey
                  WHERE local_date IS NOT NULL
                  GROUP BY participant_id, local_date HAVING COUNT(*) > 1
              ))
       FROM silver_wellness_survey""",
)
show_query(
    "Normalized survey score distributions",
    """SELECT 'fatigue_score' AS metric, MIN(fatigue_score) AS min_value,
              MAX(fatigue_score) AS max_value, COUNT(*) FILTER (WHERE fatigue_score IS NULL) AS null_values
       FROM silver_wellness_survey
       UNION ALL SELECT 'stress_score', MIN(stress_score), MAX(stress_score), COUNT(*) FILTER (WHERE stress_score IS NULL)
       FROM silver_wellness_survey
       UNION ALL SELECT 'readiness_score', MIN(readiness_score), MAX(readiness_score), COUNT(*) FILTER (WHERE readiness_score IS NULL)
       FROM silver_wellness_survey
       UNION ALL SELECT 'sleep_quality_score', MIN(sleep_quality_score), MAX(sleep_quality_score), COUNT(*) FILTER (WHERE sleep_quality_score IS NULL)
       FROM silver_wellness_survey""",
)
acceptance["Silver key nulls and duplicates are absent"] = connection.execute(
    """SELECT
           (SELECT COUNT(*) FROM silver_participants WHERE participant_id IS NULL) = 0
       AND NOT EXISTS (
           SELECT 1 FROM silver_participants GROUP BY participant_id HAVING COUNT(*) > 1
       )
       AND (SELECT COUNT(*) FROM silver_wellness_survey WHERE participant_id IS NULL OR local_date IS NULL) = 0
       AND NOT EXISTS (
           SELECT 1 FROM silver_wellness_survey WHERE local_date IS NOT NULL
           GROUP BY participant_id, local_date HAVING COUNT(*) > 1
       )"""
).fetchone()[0]

In [ ]:
# Section 6: review the human-readable report, quality rule outcomes, and traceable rejects.
display(Markdown(report_text))
show_query(
    "Quality rule outcomes",
    """SELECT rule_id, source, handling, severity, checked_count, failed_count, reason
       FROM quality_results ORDER BY rule_id, source""",
)
show_query(
    "Reject counts by rule",
    """SELECT rule_id, source, COUNT(*) AS rejected_rows
       FROM quality_rejects GROUP BY rule_id, source ORDER BY rule_id, source""",
)
show_query(
    "Reject traceability sample (raw payloads intentionally omitted)",
    """SELECT rule_id, source, row_ref, reason
       FROM quality_rejects ORDER BY rule_id, row_ref LIMIT 5""",
)
quality_results = connection.execute(
    "SELECT rule_id, failed_count FROM quality_results ORDER BY rule_id"
).fetchall()
acceptance["Five quality rules are persisted"] = len(quality_results) == 5
acceptance["Only the 14 known P999 orphans fail a quality rule"] = (
    sum(count for _, count in quality_results) == 14
    and dict(quality_results).get("SURVEY_ORPHAN_PARTICIPANT") == 14
)
acceptance["Orphan rejects retain traceable references"] = connection.execute(
    """SELECT COUNT(*) FROM quality_rejects
       WHERE rule_id = 'SURVEY_ORPHAN_PARTICIPANT' AND row_ref IS NOT NULL AND reason IS NOT NULL"""
).fetchone()[0] == 14

In [ ]:
# Section 7: validate participant-day gold grain, survey coverage, and first metric summaries.
show_query(
    "Daily gold coverage and key checks",
    """SELECT COUNT(*) AS gold_rows,
              COUNT(DISTINCT participant_id) AS participants,
              COUNT(DISTINCT local_date) AS dates,
              COUNT(*) FILTER (WHERE survey_present) AS days_with_survey,
              COUNT(*) FILTER (WHERE NOT survey_present) AS days_without_survey,
              COUNT(*) FILTER (WHERE survey_quality_flags IS NOT NULL) AS flagged_days,
              (SELECT COUNT(*) FROM (
                  SELECT participant_id, local_date FROM gold_clinical_daily
                  GROUP BY participant_id, local_date HAVING COUNT(*) > 1
              )) AS duplicate_keys
       FROM gold_clinical_daily""",
)
show_query(
    "Per-participant day coverage",
    """SELECT MIN(day_count) AS min_days_per_participant,
              MAX(day_count) AS max_days_per_participant,
              COUNT(*) FILTER (WHERE day_count <> 28) AS participants_not_at_28_days
       FROM (
           SELECT participant_id, COUNT(DISTINCT local_date) AS day_count
           FROM gold_clinical_daily GROUP BY participant_id
       )""",
)
show_query(
    "Gold score summaries (aggregates only)",
    """SELECT 'fatigue_score' AS metric, AVG(fatigue_score) AS mean_value,
              MIN(fatigue_score) AS min_value, MAX(fatigue_score) AS max_value,
              COUNT(*) FILTER (WHERE fatigue_score IS NULL) AS null_days
       FROM gold_clinical_daily
       UNION ALL SELECT 'stress_score', AVG(stress_score), MIN(stress_score), MAX(stress_score), COUNT(*) FILTER (WHERE stress_score IS NULL)
       FROM gold_clinical_daily
       UNION ALL SELECT 'readiness_score', AVG(readiness_score), MIN(readiness_score), MAX(readiness_score), COUNT(*) FILTER (WHERE readiness_score IS NULL)
       FROM gold_clinical_daily
       UNION ALL SELECT 'sleep_quality_score', AVG(sleep_quality_score), MIN(sleep_quality_score), MAX(sleep_quality_score), COUNT(*) FILTER (WHERE sleep_quality_score IS NULL)
       FROM gold_clinical_daily""",
)
show_query(
    "Single-row silver-to-gold trace",
    """SELECT s.participant_id, s.date_raw, s.fatigue_score_raw,
              s.fatigue_score AS silver_fatigue_score,
              g.fatigue_score AS gold_fatigue_score,
              s.quality_flags, g.survey_present, g.survey_row_count,
              g.survey_valid_score_count
       FROM silver_wellness_survey AS s
       JOIN gold_clinical_daily AS g
         ON s.participant_id = g.participant_id AND s.local_date = g.local_date
       WHERE s.participant_id = 'P001' AND s.local_date = DATE '2026-01-08'""",
)
gold_summary = connection.execute(
    """SELECT COUNT(*), COUNT(DISTINCT participant_id), COUNT(DISTINCT local_date),
              COUNT(*) FILTER (WHERE survey_present), COUNT(*) FILTER (WHERE NOT survey_present)
       FROM gold_clinical_daily"""
).fetchone()
acceptance["Gold contains the enrolled participant-day grid"] = gold_summary == (1400, 50, 28, 1400, 0)
acceptance["Gold key has no duplicate participant-days"] = connection.execute(
    """SELECT NOT EXISTS (
           SELECT 1 FROM gold_clinical_daily GROUP BY participant_id, local_date HAVING COUNT(*) > 1
       )"""
).fetchone()[0]
acceptance["P999 is absent from gold"] = connection.execute(
    "SELECT COUNT(*) = 0 FROM gold_clinical_daily WHERE participant_id = 'P999'"
).fetchone()[0]

In [ ]:
# Section 7a: inspect the full gold schema and representative participant-day records.
gold_inspection_connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
try:
    show_query(
        "Gold clinical daily schema",
        """SELECT name AS column_name, type AS data_type,
                  pk AS is_primary_key
           FROM pragma_table_info('gold_clinical_daily') ORDER BY cid""",
        db=gold_inspection_connection,
    )
    show_query(
        "Gold participant-day examples",
        """SELECT * FROM gold_clinical_daily
           WHERE participant_id IN ('P001', 'P002')
           QUALIFY ROW_NUMBER() OVER (
               PARTITION BY participant_id ORDER BY local_date
           ) <= 3
           ORDER BY participant_id, local_date""",
        db=gold_inspection_connection,
    )
finally:
    gold_inspection_connection.close()

In [ ]:
# Section 8: inspect the declared gold primary key and current MERGE invariants.
show_query(
    "Declared primary-key columns",
    """SELECT name AS column_name, type AS data_type, pk AS is_primary_key
       FROM pragma_table_info('gold_clinical_daily')
       WHERE pk ORDER BY cid""",
)
primary_key_columns = tuple(
    row[0] for row in connection.execute(
        "SELECT name FROM pragma_table_info('gold_clinical_daily') WHERE pk ORDER BY cid"
    ).fetchall()
)
render_table(
    "MERGE contract review",
    ["invariant", "evidence", "status"],
    [
        ("Key is participant_id + local_date", primary_key_columns,
         "PASS" if primary_key_columns == ("participant_id", "local_date") else "FAIL"),
        ("No duplicate keys after pipeline run", "Gold duplicate-key query above returned zero",
         "PASS" if acceptance["Gold key has no duplicate participant-days"] else "FAIL"),
        ("Stale-key deletion behavior", "Not exercised; INC-002 explicitly retains stale rows",
         "DEFERRED to INC-006"),
    ],
)
acceptance["Gold primary key is participant_id + local_date"] = primary_key_columns == (
    "participant_id", "local_date"
)

In [ ]:
# Section 9: test same-input rerun idempotency on an isolated copy, leaving the reviewed DB untouched.
connection.close()
sys.path.insert(0, str(ROOT / "src"))
from wearable_pipeline.pipeline import run_pipeline

table_names = [
    "raw_manifest",
    "silver_participants",
    "silver_wellness_survey",
    "quality_results",
    "quality_rejects",
    "gold_clinical_daily",
]
rerun_comparisons = []
with tempfile.TemporaryDirectory(prefix="inc002-idempotency-review-") as temporary_root:
    temporary_root = Path(temporary_root)
    baseline_dir = temporary_root / "baseline"
    rerun_dir = temporary_root / "rerun"
    (baseline_dir / "quality").mkdir(parents=True)
    (rerun_dir / "quality").mkdir(parents=True)

    baseline_db = baseline_dir / "wearable.duckdb"
    rerun_db = rerun_dir / "wearable.duckdb"
    shutil.copy2(DATABASE_PATH, baseline_db)
    shutil.copy2(DATABASE_PATH, rerun_db)
    baseline_report = baseline_dir / "quality" / "report.md"
    rerun_report = rerun_dir / "quality" / "report.md"
    shutil.copy2(REPORT_PATH, baseline_report)
    shutil.copy2(REPORT_PATH, rerun_report)

    rerun_result = run_pipeline(DATA_DIR, rerun_dir)
    baseline_connection = duckdb.connect(str(baseline_db), read_only=True)
    rerun_connection = duckdb.connect(str(rerun_db), read_only=True)
    try:
        for table_name in table_names:
            before_rows = baseline_connection.execute(
                f'SELECT * FROM "{table_name}" ORDER BY ALL'
            ).fetchall()
            after_rows = rerun_connection.execute(
                f'SELECT * FROM "{table_name}" ORDER BY ALL'
            ).fetchall()
            rerun_comparisons.append((table_name, len(before_rows), before_rows == after_rows))
    finally:
        baseline_connection.close()
        rerun_connection.close()

    report_identical = baseline_report.read_bytes() == rerun_report.read_bytes()
    rerun_id_identical = rerun_result.run_id == run_id

render_table(
    "Same-input rerun comparison",
    ["table", "rows before", "contents identical after MERGE/rebuild", "status"],
    [(*row, "PASS" if row[2] else "FAIL") for row in rerun_comparisons],
)
render_table(
    "Rerun determinism",
    ["check", "result", "status"],
    [
        ("Run ID unchanged", rerun_id_identical, "PASS" if rerun_id_identical else "FAIL"),
        ("Quality report bytes unchanged", report_identical, "PASS" if report_identical else "FAIL"),
    ],
)
acceptance["Rerun preserves every table's logical contents"] = bool(rerun_comparisons) and all(
    row[2] for row in rerun_comparisons
)
acceptance["Rerun preserves the deterministic run id and report"] = (
    rerun_id_identical and report_identical
)

In [ ]:
# Section 10: compile query-backed review evidence; changed-input updates and stale-key removal are not exercised here.
final_connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
final_counts = {
    table_name: final_connection.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    for table_name in table_names
}
final_connection.close()
expected_counts = {
    "silver_participants": 50,
    "silver_wellness_survey": 1414,
    "quality_results": 5,
    "quality_rejects": 14,
    "gold_clinical_daily": 1400,
}
count_checks = [
    (table_name, expected, final_counts[table_name], final_counts[table_name] == expected)
    for table_name, expected in expected_counts.items()
]
acceptance["Persisted table counts match this profiled dataset"] = all(
    passed for _, _, _, passed in count_checks
)

render_table(
    "Expected versus observed row counts",
    ["table", "expected for this dataset", "observed", "status"],
    [
        (table_name, expected, observed, "PASS" if passed else "FAIL")
        for table_name, expected, observed, passed in count_checks
    ],
)
acceptance_rows = [
    (name, "PASS" if result is True else "FAIL" if result is False else "NOT RUN")
    for name, result in acceptance.items()
]
acceptance_rows.extend([
    ("Changed-input MERGE updates exactly the affected gold row", "NOT RUN in this review notebook; covered by INC-002 tests"),
    ("Stale gold keys are deleted", "DEFERRED to INC-006 by the INC-002 contract"),
])
render_table("INC-002 review summary", ["acceptance check", "status"], acceptance_rows)
print("Review is observational; only the isolated temporary database copy was rerun.")